# PSU efficiency analysis

> Version 2

Revision of the PSU analysis presented in the IMC'25 paper (Section 9)

Change log:
- v2:
  - **Fixed the interpolation function**  
  The original anaylsis used the numpy interpolation function, but that one does not extrapolate outside of the input data range, which is necessary in our case, as the load of many PSUs is smaller than the first set point for the interpolation.  
  This fix **significantly impacts** the outcome of the PSU right-sizing analysis. In particular, the cost of oversizing increases from 1% to 4% if we would replace all PSUs by the largest ones.
  - **Fixed the PSU load computation**   
  The PSU load was computed using the ratio of the median power measurements with the PSU capacity. This does not match with the standard definition of the PSU load, which is `P_out/PSU_capacity`, while our power measurements measure `P_in`, not `P_out`.
  - **Added modular analysis for efficiencies >1**  
  Some PSU efficiencies would evolve to be larger than 1, which makes no physical sense. This is because of the approximation we make in the PSU efficiency estimation. To address this, we added some modular capping of the efficiency values. The options are:
    - No capping,
    - Capping to 1  _< Default_
    - Capping to the Titanium curve.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
from scipy.optimize import fsolve
from scipy import interpolate

import plotly.graph_objects as go

## Initialization

In [2]:

# Select the different output format settings

# PaperPlot = True
PaperPlot = False
if PaperPlot:
    output_format = 'IMC'
else:
    output_format = 'online'

if output_format == 'online':
    font_size_px = 14
    linewidth_px = 512
    landscapewidth_px = 2*linewidth_px
    plot_path = None
    
    out_path = Path('output/online/figures')

if output_format == 'IMC':
    font_size_pt = 7
    offset = 5 # to compensate for the rounding of unit conversions
    linewidth_pt = 241 - offset  
    landscapewidth_pt = 506 - offset
    
    # 1pt = 1.333px
    font_size_px = int(font_size_pt*1.333)+1
    linewidth_px = int(linewidth_pt*1.333)+1
    landscapewidth_px = int(landscapewidth_pt*1.333)+1

    out_path = Path('output/2025_IMC/figures')

# Create the output directory if don't exist
Path(out_path).mkdir(parents=True, exist_ok=True)

# Input data
input_path = Path('input')

# Default plot layout
default_layout = {
    "title":None,
    "width":linewidth_px,
    "height":200,
    "font":{"size":font_size_px},
    "yaxis":{'title':{'font':{'size':font_size_px}}},
    "xaxis":{'title':{'font':{'size':font_size_px},
                      'text':'Time [ s ]'}}
}


## Efficient curve and 80 Plus standards

In [3]:
# ===============
# Import PFE data and derive the numerical models 
# for the different standard by adding a constant 
# to the PFE efficiency curve
# ===============

pfe_file = Path(input_path,'pfe-efficiency-curve.csv')
df_pfe = pd.read_csv(pfe_file)
df_pfe['load_%'] = (df_pfe['load_W'] / 600)*100

# .. Model is linear interpolation between the colected points
# => The numpy function does not extrapolate outside of the input data range, 
#    which we need for our analysis.
# def eff_interp(x):
#     return np.interp(x,df_pfe['load_%']/100,df_pfe['efficiency_%']/100)

# Setup the linear interpolation between the collected points
# (which also extrapolate linearly outside of the range)
eff_interp = interpolate.interp1d(
    df_pfe['load_%']/100, 
    df_pfe['efficiency_%']/100,
    fill_value="extrapolate")

marker_opacity = 1
marker_size = 9
marker_border_size = 1
marker_line = dict(
    width=marker_border_size,
    color='black'
)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x = [10,20,50,100],
        y = [90,94,96,91],
        mode='markers',
        name='Titanium',
        marker=dict(
            symbol=19, 
            line=marker_line,
            size=marker_size, 
            color='rgb(121, 121, 130)', 
            ),
    )
)
fig.add_trace(
    go.Scatter(
        x = [20,50,100],
        y = [90,94,91],
        mode='markers',
        name='Platinum',
        marker=dict(symbol=0, 
            line=marker_line,size=marker_size, color='rgb(209, 208, 206)', ),
    )
)
fig.add_trace(
    go.Scatter(
        x = [20,50,100],
        y = [88,92,88],
        mode='markers',
        name='Gold',
        marker=dict(symbol=2,
            line=marker_line, size=marker_size, color='rgb(255, 215, 0)', opacity=marker_opacity),
    )
)
fig.add_trace(
    go.Scatter(
        x = [20,50,100],
        y = [85,89,85],
        mode='markers',
        name='Silver',
        marker=dict(symbol=4,
            line=marker_line, size=marker_size, color='rgb(192, 192, 192)', opacity=marker_opacity),
    )
)
fig.add_trace(
    go.Scatter(
        x = [20,50,100],
        y = [81,85,81],
        mode='markers',
        name='Bronze',
        marker=dict(symbol=3,
            line=marker_line, size=marker_size, color='rgb(205, 127, 50)', opacity=marker_opacity),
    )
)

fig.add_trace(
    go.Scatter(
        x = df_pfe['load_%'],
        y = df_pfe['efficiency_%'],
        mode='lines',
        name='PFE600',
        marker=dict(
            color='black', 
        )
    )
)

 # y axis title
ytitle = go.layout.Annotation(
        x=-0.01,
        y=1.15,
        xref="paper",
        yref="paper",
        text="Efficiency (%)",
        showarrow=False,
        xanchor='left'
    )

# Define the custom layout options
custom_layout = dict(
    xaxis=dict(
        title=dict(
            text='Power load (%)',
            font={'size':font_size_px}
        ),
        range=[-5,105],
    ),
    yaxis=dict(
        title=None,
        range=[80,100],
    ),
    legend=dict(
        orientation='v'
    ),
    annotations=[ytitle],
    margin=dict(l=0, r=0, t=25, b=0),
)
# Combine with the defaults and apply
layout = default_layout.copy()
layout.update(custom_layout)
fig.update_layout(layout)

fig.show()
fig.write_image(out_path/'80Plus-curves.pdf')

In [4]:
# ===============
# Interpolation functions for the different 80Plus standards
# ===============

# Function f(x) for 80PLUS Titanium standard
def t(x):
    const = 0.94 - eff_interp(0.20) + 0.02
    return eff_interp(x) + const
    # return -0.2083 * x**2 + 0.2125 * x + 0.90583

# Function f(x) for 80PLUS Platinum standard
def p(x):
    const = 0
    return eff_interp(x) + const
    # return -0.2417 * (x)**2 + 0.3025 * (x) + 0.84917

# Function f(x) for 80PLUS Gold standard
def g(x):
    const = 0.92 - eff_interp(0.50)
    return eff_interp(x) + const
    # return -0.2667 * (x)**2 + 0.32 * (x) + 0.8267

# Function f(x) for 80PLUS Silver standard
def s(x):
    const = 0.89 - eff_interp(0.50)
    return eff_interp(x) + const
    # return -0.2667 * (x)**2 + 0.32 * (x) + 0.7967

# Function f(x) for 80PLUS Bronze standard
def b(x):
    const = 0.85 - eff_interp(0.50)
    return eff_interp(x) + const
    # return -0.2667 * (x)**2 + 0.32 * (x) + 0.7567

# Function f(x) for an 80PLUS-like standard
# with custom constent term
def custom_eff(x,const):
    return eff_interp(x) + const

In [5]:
# ===============
# Read the PSU efficiency data
# ===============

# Setup the function of the numerical solver for the load
# load_guess = 0.1 # 10% load initial guess for the solver
def load_equation(load,capacity,p_in):
    # P_out = capacity * load = p_in * eff_interp(load)
    return capacity * load - p_in * eff_interp(load)

def solve_for_load(row,PSU,label_capacity):
    load_guess = 0.1 # 10% load initial guess for the solver
    return fsolve(load_equation,load_guess, args=(row[label_capacity],row['median_power_PSU'+PSU]))[0]

def read_src_data():
        
    src_file = Path(input_path,'psu-efficiency.csv')
    df = pd.read_csv(src_file)

    # Numerically solve to get the PSU loads from the P_in values
    for PSU in ['1','2']:
        label_load = 'load_PSU' + PSU
        df[label_load] = df.apply(solve_for_load, args=(PSU,'PSU_capacity',), axis=1)           

    return df

df = read_src_data()


df


C:\Users\Ibrahim\AppData\Local\Temp\ipykernel_19628\1817319691.py:13: RuntimeWarning:

The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.



,router_id,router_model,mean_power_PSU1,mean_power_PSU2,median_power_PSU1,median_power_PSU2,PSU_capacity,efficiency_PSU1,efficiency_PSU2,inlet_temperature,load_PSU1,load_PSU2
0,swiFN1,NCS-55A1-24H,191.171476,180.411720,191.020644,180.411000,1100,0.808791,0.839479,NaN,0.156268,0.146749
1,swiWY1,NCS-55A1-24H,194.310960,165.750012,194.311000,165.499000,1100,0.878525,0.922219,NaN,0.159239,0.133289
2,swiES3,NCS-55A1-24H,191.890173,166.704280,191.782000,164.063000,1100,0.938731,0.857143,NaN,0.156954,0.132000
3,swiUS1,NCS-55A1-24H,180.736509,167.841851,180.950930,167.873916,1100,0.872845,0.790150,NaN,0.147239,0.135425
4,swiTR3,NCS-55A1-24H,212.974344,180.248504,213.636000,180.261268,1100,0.883041,0.869099,NaN,0.176869,0.146612
...,...,...,...,...,...,...,...,...,...,...,...,...
102,swiWB1,NCS-55A1-48Q6H,186.161235,170.581386,185.614968,171.750000,1100,0.925602,0.812227,NaN,0.151409,0.138907
103,swiUN540,N540-24Z8Q2C-M,79.681341,79.645633,79.698000,79.665019,400,0.679551,0.693708,NaN,0.181931,0.181848
104,swiOR1,WS-C6504-E,164.589201,172.267520,164.781000,169.380210,2700,NaN,NaN,NaN,0.050601,0.052077
105,swiOT2,8201-24H8FH,110.206985,156.885401,112.000000,158.175901,2000,0.545455,0.645652,26.0,0.046264,0.066433


## Plot eff=f(load)

In [6]:
# ===============
# Plot the PSU conversion efficiency data
# ===============

df = read_src_data()

fig = go.Figure()

marker_opacity = 0.5
marker_size = 9
marker_border_size = 1
marker_line = dict(
    width=marker_border_size,
    color='black'
)

# Combine the data from both PSU
loads = df['load_PSU1'].tolist()
loads.append(df['load_PSU2'].tolist())
efficiencies = df['efficiency_PSU1'].tolist()
efficiencies.append(df['efficiency_PSU2'].tolist())


fig.add_trace(
    go.Scatter(
        x = df_pfe['load_%']/100,
        y = df_pfe['efficiency_%']/100 + 0.94 - eff_interp(0.20) + 0.02,
        mode='lines',
        name='Titanium',
        marker=dict(
            color='rgb(121, 121, 130)', 
        ),
        showlegend=False
    )
)
fig.add_trace(
    go.Scatter(
        x = [0.2],
        y = [0.96],
        mode='markers',
        name='Titanium',
        marker=dict(
            symbol=19, 
            line=marker_line,
            size=marker_size, 
            color='rgb(121, 121, 130)', 
            ),
        showlegend=False
    )
)
fig.add_trace(
    go.Scatter(
        x = df_pfe['load_%']/100,
        y = df_pfe['efficiency_%']/100,
        mode='lines',
        name='Platinum',
        marker=dict(
            color='rgb(209, 208, 206)', 
        ),
        showlegend=False
    )
)
fig.add_trace(
    go.Scatter(
        x = [0.2],
        y = [0.922],
        mode='markers',
        name='Platinum',
        marker=dict(symbol=0, 
            line=marker_line,size=marker_size, color='rgb(209, 208, 206)', ),
        showlegend=False
    )
)
fig.add_trace(
    go.Scatter(
        x = df_pfe['load_%']/100,
        y = df_pfe['efficiency_%']/100 + 0.92 - eff_interp(0.50),
        mode='lines',
        name='Gold',
        marker=dict(
            color='rgb(255, 215, 0)', 
        ),
        showlegend=False
    )
)
fig.add_trace(
    go.Scatter(
        x = [0.20],
        y = [0.899],
        mode='markers',
        name='Gold',
        marker=dict(symbol=2,
            line=marker_line, size=marker_size, color='rgb(255, 215, 0)', 
            ),
        showlegend=False
    )
)
fig.add_trace(
    go.Scatter(
        x = df_pfe['load_%']/100,
        y = df_pfe['efficiency_%']/100 + 0.89 - eff_interp(0.50),
        mode='lines',
        name='Silver',
        marker=dict(
            color='rgb(192, 192, 192)', 
        ),
        showlegend=False
    )
)
fig.add_trace(
    go.Scatter(
        x = [0.20],
        y = [0.87],
        mode='markers',
        name='Silver',
        marker=dict(symbol=4,
            line=marker_line, size=marker_size, color='rgb(192, 192, 192)', 
            ),
        showlegend=False
    )
)
fig.add_trace(
    go.Scatter(
        x = df_pfe['load_%']/100,
        y = df_pfe['efficiency_%']/100 + 0.85 - eff_interp(0.50),
        mode='lines',
        name='Bronze',
        marker=dict(
            color='rgb(205, 127, 50)', 
        ),
        showlegend=False
    )
)
fig.add_trace(
    go.Scatter(
        x = [0.20],
        y = [0.83],
        mode='markers',
        name='Bronze',
        marker=dict(symbol=3,
            line=marker_line, size=marker_size, color='rgb(205, 127, 50)', 
            ),
        showlegend=False
    )
)


fig.add_trace(
    go.Scatter(
        x = loads,
        y = efficiencies,
        mode='markers',
        marker=dict(
            opacity=marker_opacity,
        ),
        showlegend=False
    )
)


 # y axis title
ytitle = go.layout.Annotation(
        x=-0.01,
        y=1.15,
        xref="paper",
        yref="paper",
        text="Efficiency (%)",
        showarrow=False,
        xanchor='left'
    )

# Define the custom layout options
custom_layout = dict(
    xaxis=dict(
        title=dict(
            text='Power load (%)',
            font={'size':font_size_px}
        ),
        range=[0.03,0.23],
        tickmode = 'array',
        tickvals = [0.05, 0.1, 0.15, 0.2],
        ticktext = [5, 10, 15, 20]
    ),
    yaxis=dict(
        title=None,
        range=[0.50,1.05],
    ),
    legend=dict(
        orientation='v'
    ),
    width=0.3*landscapewidth_px,
    annotations=[ytitle],
    margin=dict(l=0, r=0, t=25, b=0),
)
# Combine with the defaults and apply
layout = default_layout.copy()
layout.update(custom_layout)
fig.update_layout(layout)


fig.show()
fig.write_image(out_path/'efficiency_all.pdf')

C:\Users\Ibrahim\AppData\Local\Temp\ipykernel_19628\1817319691.py:13: RuntimeWarning:

The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.



In [7]:
# ===============
# Highlight the data from a given router model
# ===============


# Filter for the router model of interest

# model_id = '8201-32FH'
# model_id = 'NCS-55A1-24H'
model_id = 'ASR-920-24SZ-M'
# model_id = '8201-24H8FH'
tmp = df.loc[df['router_model']== model_id]

# Combine the data from both PSU
loads = tmp['load_PSU1'].tolist()
loads.append(tmp['load_PSU2'].tolist())
efficiencies = tmp['efficiency_PSU1'].tolist()
efficiencies.append(tmp['efficiency_PSU2'].tolist())

fig.add_trace(
    go.Scatter(
        x = loads,
        y = efficiencies,
        mode='markers',
        marker=dict(
            color='red', 
            opacity=marker_opacity,
        ),
        showlegend=False
    )
)

# Define the custom layout options
custom_layout = dict(
    width=0.2*landscapewidth_px,
)
# Combine with the defaults and apply
# layout = default_layout.copy() <- Don't copy! We want to update from the previous plot
layout.update(custom_layout)
fig.update_layout(layout)

fig.write_image(out_path/str('efficiency_'+model_id+'.pdf'))

fig.show()

# Optimizing PSUs

In the following, we look at different ways we could try to optimize the power conversion efficiency of PSUs. In order to do that, we make two important assumptions:

1. All PSUs are assumed to have a power efficiency curve shaped like the one of the PFE PSU, shifted by some constant offset. 
2. For each PSU, we have one (load,efficiency) data point, which allows to calibrate the offset to the PFE curve. 

Following those assumptions lead to PSU with efficiencies larger than 1 in certain cases, which, naturally, does not make much sense. We can treat this problem a number of ways; we consider the following three.

1. `baseline` : Do not do anything; accept the non-physical approximation that some efficiencies are larger than 1.
2. `max_1` : Cap all efficiency values at 1.
3. `max_titanium` : Cap all efficiency values at the Titanium curve.

In the cell below, you can select which option to use in the analysis.

> ===  
> Effects
> - Capping the values at 1 has little effects, as the ocurences of efficiencies larger than one are rare in the dataset.
> - Capping to the Titanium curve has a more significant effect, as expected.   
> 
> ===

In [8]:
analysis_options = [
    'baseline',
    'max_1',
    'max_titanium'
]
analysis = analysis_options[1]

def analysis_print(analysis):
    print('''==============
Analysis type: {}
=============='''.format(analysis))
    return

analysis_print(analysis)


Analysis type: max_1


## How much would we save with better PSUs?

That analysis is not affected by the clipping of the efficiency values to either 1 or the titanium curve, because it quantifies the savings from raising the efficiency values up to the different standards. Efficiency values above the standards do not affect the results.

In [9]:
df = read_src_data()
tex_output = ''

# Compute the theoretic efficiency for different standards
for standard in [b,s,g,p,t]:
    for PSU in ['1','2']:
        # .. compute the efficiency
        label_eff = 'efficiency_PSU' + PSU + '_' + standard.__name__
        df[label_eff] = df['load_PSU' + PSU].apply(standard)
        df[label_eff] = df[[label_eff,'efficiency_PSU' + PSU]].max(axis=1)

        # .. derive the resulting power draw
        label_power = 'median_power_PSU' + PSU + '_' + standard.__name__
        df[label_power] = df['median_power_PSU' + PSU] * df['efficiency_PSU' + PSU] / df[label_eff]

        # .. compute the correspinding savings
        label_savings_abs = 'savings_abs_PSU' + PSU + '_' + standard.__name__
        label_savings_rel = 'savings_rel_PSU' + PSU + '_' + standard.__name__
        df[label_savings_abs] = df['median_power_PSU' + PSU] - df[label_power] 
        df[label_savings_rel] = df[label_savings_abs] / df['median_power_PSU' + PSU] 

    # .. Compute the gains
    savings_total_abs = df['savings_abs_PSU1_'+standard.__name__].sum(axis=0) +  df['savings_abs_PSU2_'+standard.__name__].sum(axis=0) 
    savings_total_rel = savings_total_abs / (df['median_power_PSU1'].sum(axis=0) + df['median_power_PSU2'].sum(axis=0))

    print('For',standard.__name__)
    print("{}\\% ({} W)\n".format(
        int(100*savings_total_rel),
        int(savings_total_abs)
        ))
    
    tex_output += '& {}\\% ({} W)'.format(
        int(100*savings_total_rel),
        int(savings_total_abs)
        )
tex_output += '\\\\'
df

print(tex_output)

C:\Users\Ibrahim\AppData\Local\Temp\ipykernel_19628\1817319691.py:13: RuntimeWarning:

The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.



For b
1\% (357 W)

For s
2\% (611 W)

For g
3\% (821 W)

For p
4\% (1012 W)

For t
6\% (1402 W)

& 1\% (357 W)& 2\% (611 W)& 3\% (821 W)& 4\% (1012 W)& 6\% (1402 W)\\


## How much would we save with better-sized PSUs?

In [10]:
df = read_src_data()
psu_capacities = sorted(df['PSU_capacity'].unique())

pd.set_option('display.max_columns', None)

overprovision_factor = [1,2,3]

def set_min_capacity(row, capacity_options, overprovision_factor):
    for c in sorted(capacity_options):
        if ((row['median_power_PSU1'] > c/overprovision_factor) or
            (row['median_power_PSU2'] > c/overprovision_factor)):
            continue
        else:
            return c
        
def set_new_capacity(row, capacity):
    if (row['min_capacity'] > capacity):
        return row['min_capacity']
    else:
        return capacity
    
def get_new_efficiency(row, PSU, label_load,analysis):
    '''
    Compute the offset between the original efficiency at the original load 
    with the efficiency given by the PFE curve. 
    -> That offset is assumed constant. 
    Using that offset, we can derive the new efficiency at the new load value
    by interpoleting from the PFE curve and adding the offset.
    '''
    offset = row['efficiency_PSU' + PSU] - eff_interp(row['load_PSU' + PSU])
    new_efficiency = custom_eff(row[label_load],offset)
    if analysis == 'baseline':
        return new_efficiency
    elif analysis == 'max_1':
        return min(new_efficiency,1)
    elif analysis == 'max_titanium':
        return min(new_efficiency,t(row['load_PSU' + PSU]))
    return custom_eff(row[label_load],offset)   

# Log
analysis_print(analysis)
    
for k in overprovision_factor:

    tex_output = ''
    print('k = {}'.format(k))
    
    # .. define the minimal capacity required per router
    df['min_capacity'] = df.apply(set_min_capacity, args=(psu_capacities, k), axis=1)

    # .. set the different cases of smallest considered capacities
    for capacity in psu_capacities:

        # .. compute the min capacity per router per case
        label_capacity = 'capacity_'+str(capacity)+'+'
        df[label_capacity] = df.apply(set_new_capacity, args=(capacity, ), axis=1)
        
        for PSU in ['1','2']:
            # .. compute the corresponding load
            label_load = 'load_PSU' + PSU + '_' + label_capacity
            df[label_load] = df.apply(solve_for_load, args=(PSU,label_capacity,), axis=1)           

            # .. compute the corresponding efficiency
            label_eff = 'efficiency_PSU' + PSU + '_' + label_capacity
            df[label_eff] = df.apply(get_new_efficiency, args=(PSU,label_load,analysis,), axis=1)

            # .. derive the resulting power draw
            label_power = 'median_power_PSU' + PSU + '_' + label_capacity
            df[label_power] = df['median_power_PSU' + PSU] * df['efficiency_PSU' + PSU] / df[label_eff]

            # .. compute the correspinding savings
            label_savings_abs = 'savings_abs_PSU' + PSU + '_' + label_capacity
            label_savings_rel = 'savings_rel_PSU' + PSU + '_' + label_capacity
            df[label_savings_abs] = df['median_power_PSU' + PSU] - df[label_power] 
            df[label_savings_rel] = df[label_savings_abs] / df['median_power_PSU' + PSU] 

        # .. Compute the total gains
        savings_total_abs = df['savings_abs_PSU1_' + label_capacity].sum(axis=0) +  df['savings_abs_PSU2_' + label_capacity].sum(axis=0) 
        savings_total_rel = savings_total_abs / (df['median_power_PSU1'].sum(axis=0) + df['median_power_PSU2'].sum(axis=0))

        print('For $',label_capacity,'$')
        print("{}\\% ({} W)\n".format(
            int(savings_total_rel*100),
            int(savings_total_abs)
            ))
        

        tex_output += '& {}\\% ({} W)'.format(
                int(100*savings_total_rel),
                int(savings_total_abs)
                )

    tex_output += '\\\\'
    print(tex_output)

    display(df)

C:\Users\Ibrahim\AppData\Local\Temp\ipykernel_19628\1817319691.py:13: RuntimeWarning:

The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.



Analysis type: max_1
k = 1


C:\Users\Ibrahim\AppData\Local\Temp\ipykernel_19628\1817319691.py:13: RuntimeWarning:

The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.

C:\Users\Ibrahim\AppData\Local\Temp\ipykernel_19628\1817319691.py:13: RuntimeWarning:

The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.



For $ capacity_250+ $
3\% (697 W)



C:\Users\Ibrahim\AppData\Local\Temp\ipykernel_19628\1817319691.py:13: RuntimeWarning:

The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.



...

C:\Users\Ibrahim\AppData\Local\Temp\ipykernel_19628\1817319691.py:13: RuntimeWarning:

The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.



For $ capacity_400+ $
1\% (322 W)



C:\Users\Ibrahim\AppData\Local\Temp\ipykernel_19628\1817319691.py:13: RuntimeWarning:

The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.

C:\Users\Ibrahim\AppData\Local\Temp\ipykernel_19628\1817319691.py:13: RuntimeWarning:

The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.



For $ capacity_750+ $
0\% (174 W)

For $ capacity_1100+ $
-1\% (-254 W)

For $ capacity_2000+ $
-4\% (-853 W)

For $ capacity_2700+ $
-4\% (-1047 W)

& 2\% (499 W)& 1\% (322 W)& 0\% (174 W)& -1\% (-254 W)& -4\% (-853 W)& -4\% (-1047 W)\\


,router_id,router_model,mean_power_PSU1,mean_power_PSU2,median_power_PSU1,median_power_PSU2,PSU_capacity,efficiency_PSU1,efficiency_PSU2,inlet_temperature,load_PSU1,load_PSU2,min_capacity,capacity_250+,load_PSU1_capacity_250+,efficiency_PSU1_capacity_250+,median_power_PSU1_capacity_250+,savings_abs_PSU1_capacity_250+,savings_rel_PSU1_capacity_250+,load_PSU2_capacity_250+,efficiency_PSU2_capacity_250+,median_power_PSU2_capacity_250+,savings_abs_PSU2_capacity_250+,savings_rel_PSU2_capacity_250+,capacity_400+,load_PSU1_capacity_400+,efficiency_PSU1_capacity_400+,median_power_PSU1_capacity_400+,savings_abs_PSU1_capacity_400+,savings_rel_PSU1_capacity_400+,load_PSU2_capacity_400+,efficiency_PSU2_capacity_400+,median_power_PSU2_capacity_400+,savings_abs_PSU2_capacity_400+,savings_rel_PSU2_capacity_400+,capacity_750+,load_PSU1_capacity_750+,efficiency_PSU1_capacity_750+,median_power_PSU1_capacity_750+,savings_abs_PSU1_capacity_750+,savings_rel_PSU1_capacity_750+,load_PSU2_capacity_750+,efficiency_PSU2_capacity_750+,median_power_PSU2_capacity_750+,savings_abs_PSU2_capacity_750+,savings_rel_PSU2_capacity_750+,capacity_1100+,load_PSU1_capacity_1100+,efficiency_PSU1_capacity_1100+,median_power_PSU1_capacity_1100+,savings_abs_PSU1_capacity_1100+,savings_rel_PSU1_capacity_1100+,load_PSU2_capacity_1100+,efficiency_PSU2_capacity_1100+,median_power_PSU2_capacity_1100+,savings_abs_PSU2_capacity_1100+,savings_rel_PSU2_capacity_1100+,capacity_2000+,load_PSU1_capacity_2000+,efficiency_PSU1_capacity_2000+,median_power_PSU1_capacity_2000+,savings_abs_PSU1_capacity_2000+,savings_rel_PSU1_capacity_2000+,load_PSU2_capacity_2000+,efficiency_PSU2_capacity_2000+,median_power_PSU2_capacity_2000+,savings_abs_PSU2_capacity_2000+,savings_rel_PSU2_capacity_2000+,capacity_2700+,load_PSU1_capacity_2700+,efficiency_PSU1_capacity_2700+,median_power_PSU1_capacity_2700+,savings_abs_PSU1_capacity_2700+,savings_rel_PSU1_capacity_2700+,load_PSU2_capacity_2700+,efficiency_PSU2_capacity_2700+,median_power_PSU2_capacity_2700+,savings_abs_PSU2_capacity_2700+,savings_rel_PSU2_capacity_2700+
0,swiFN1,NCS-55A1-24H,191.171476,180.411720,191.020644,180.411000,1100,0.808791,0.839479,NaN,0.156268,0.146749,750,750,0.236817,0.838726,184.203014,6.817630,0.035691,0.223148,0.872389,173.605196,6.805804,0.037724,750,0.236817,0.838726,184.203014,6.817630,0.035691,0.223148,0.872389,173.605196,6.805804,0.037724,750,0.236817,0.838726,184.203014,6.817630,0.035691,0.223148,0.872389,173.605196,6.805804,0.037724,1100,0.156268,0.808791,191.020644,0.000000e+00,0.000000e+00,0.146749,0.839479,180.411000,0.000000,0.000000,2000,0.081197,0.759050,203.538273,-12.517629,-0.065530,0.076389,0.791558,191.333275,-10.922275,-0.060541,2700,0.059070,0.743854,207.696461,-16.675817,-0.087299,0.055632,0.777301,194.842483,-14.431483,-0.079992
1,swiWY1,NCS-55A1-24H,194.310960,165.750012,194.311000,165.499000,1100,0.878525,0.922219,NaN,0.159239,0.133289,750,750,0.241013,0.907327,188.142759,6.168241,0.031744,0.203737,0.959594,159.053052,6.445948,0.038949,750,0.241013,0.907327,188.142759,6.168241,0.031744,0.203737,0.959594,159.053052,6.445948,0.038949,750,0.241013,0.907327,188.142759,6.168241,0.031744,0.203737,0.959594,159.053052,6.445948,0.038949,1100,0.159239,0.878525,194.311000,0.000000e+00,0.000000e+00,0.133289,0.922219,165.499000,0.000000,0.000000,2000,0.082695,0.828231,206.110456,-11.799456,-0.060725,0.069694,0.878541,173.726945,-8.227945,-0.049716,2700,0.060141,0.812740,210.038883,-15.727883,-0.080942,0.050831,0.865586,176.327163,-10.828163,-0.065427
2,swiES3,NCS-55A1-24H,191.890173,166.704280,191.782000,164.063000,1100,0.938731,0.857143,NaN,0.156954,0.132000,750,750,0.237788,0.968404,185.905572,5.876428,0.030641,0.201809,0.894667,157.181925,6.881075,0.041942,750,0.237788,0.968404,185.905572,5.876428,0.030641,0.201809,0.894667,157.181925,6.881075,0.041942,750,0.237788,0.968404,185.905572,5.876428,0.030641,0.201809,0.894667,157.181925,6.881075,0.041942,1100,0.156954,0.938731,191.782000,0.000000e+00,0.000000e+00

## How much would we save by loading only one PSU instead of two?

In [11]:
# Load the data
df = read_src_data()
tex_output = ''

analysis_print(analysis)

# .. compute the total load
df['total_power_out'] = df['median_power_PSU1']*df['efficiency_PSU1'] + df['median_power_PSU2']*df['efficiency_PSU2']
df['total_load'] = df['total_power_out'].div(df['PSU_capacity'])

# .. compute the efficiency for that load if running on only one of the PSUs
for PSU in ['1','2']:
    label_eff = 'efficiency_PSU' + PSU + '_total_load' 
    df[label_eff] = df.apply(get_new_efficiency, args=(PSU,'total_load',analysis,), axis=1)
# .. take the max
df['max_efficiency'] = df[['efficiency_PSU1_total_load','efficiency_PSU2_total_load']].max(axis=1)
# .. get the resulting total power in
df['total_power_in'] = df['total_power_out'] / df['max_efficiency']

# .. compute the savings per router
df['savings_abs'] = (df['median_power_PSU1']+ df['median_power_PSU2']) - df['total_power_in']
df['savings_rel'] = df['savings_abs'] / (df['median_power_PSU1']+ df['median_power_PSU2'])

# .. compute the global savings
savings_total_abs = df['savings_abs'].sum(axis=0)
savings_total_rel = savings_total_abs / (df['median_power_PSU1'].sum(axis=0) + df['median_power_PSU2'].sum(axis=0))


print('For using only one PSU')
print("{}\\% ({} W)\n".format(
    int(savings_total_rel*100),
    int(savings_total_abs)
    ))

tex_output += '& {}\\% ({} W)'.format(
        int(100*savings_total_rel),
        int(savings_total_abs)
        )

print(tex_output)

df

Analysis type: max_1
For using only one PSU
5\% (1130 W)

& 5\% (1130 W)


C:\Users\Ibrahim\AppData\Local\Temp\ipykernel_19628\1817319691.py:13: RuntimeWarning:

The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.



,router_id,router_model,mean_power_PSU1,mean_power_PSU2,median_power_PSU1,median_power_PSU2,PSU_capacity,efficiency_PSU1,efficiency_PSU2,inlet_temperature,load_PSU1,load_PSU2,total_power_out,total_load,efficiency_PSU1_total_load,efficiency_PSU2_total_load,max_efficiency,total_power_in,savings_abs,savings_rel
0,swiFN1,NCS-55A1-24H,191.171476,180.411720,191.020644,180.411000,1100,0.808791,0.839479,NaN,0.156268,0.146749,305.947134,0.278134,0.842789,0.878598,0.878598,348.222031,23.209613,0.062487
1,swiWY1,NCS-55A1-24H,194.310960,165.750012,194.311000,165.499000,1100,0.878525,0.922219,NaN,0.159239,0.133289,323.333346,0.293939,0.912244,0.971484,0.971484,332.824224,26.985776,0.075000
2,swiES3,NCS-55A1-24H,191.890173,166.704280,191.782000,164.063000,1100,0.938731,0.857143,NaN,0.156954,0.132000,320.657109,0.291506,0.973480,0.907105,0.973480,329.392705,26.452295,0.074337
3,swiUS1,NCS-55A1-24H,180.736509,167.841851,180.950930,167.873916,1100,0.872845,0.790150,NaN,0.147239,0.135425,290.587640,0.264171,0.910422,0.835413,0.910422,319.179231,29.645615,0.084987
4,swiTR3,NCS-55A1-24H,212.974344,180.248504,213.636000,180.261268,1100,0.883041,0.869099,NaN,0.176869,0.146612,345.314169,0.313922,0.909058,0.911140,0.911140,378.991512,14.905755,0.037842
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102,swiWB1,NCS-55A1-48Q6H,186.161235,170.581386,185.614968,171.750000,1100,0.925602,0.812227,NaN,0.151409,0.138907,311.305539,0.283005,0.962616,0.856874,0.962616,323.395348,33.969619,0.095056
103,swiUN540,N540-24Z8Q2C-M,79.681341,79.645633,79.698000,79.665019,400,0.679551,0.693708,NaN,0.181931,0.181848,109.423071,0.273558,0.699917,0.714114,0.714114,153.229178,6.133841,0.038490
104,swiOR1,WS-C6504-E,164.589201,172.267520,164.781000,169.380210,2700,NaN,NaN,NaN,0.050601,0.052077,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
105,swiOT2,8201-24H8FH,110.206985,156.885401,112.000000,158.175901,2000,0.545455,0.645652,26.0,0.046264,0.066433,163.217524,0.081609,0.569730,0.656075,0.656075,248.778823,21.397078,0.079197


## How much would we save by loading only one PSU instead of two AND that one be of better quality?

In [12]:
analysis_print(analysis)

# [continuing from the previous df]
tex_output = ''

# Compute the theoretic efficiency for different standards
for standard in [b,s,g,p,t]:
    # .. compute the efficiency
    label_eff = 'efficiency_' + standard.__name__
    df[label_eff] = df['total_load'].apply(standard)
    df[label_eff] = df[[label_eff,'max_efficiency']].max(axis=1)

    # .. derive the resulting power draw
    label_power = 'median_power_' + standard.__name__
    df[label_power] = df['total_power_in'] * df['max_efficiency'] / df[label_eff]

    # .. compute the savings per router
    label_savings_abs = 'savings_abs_' + standard.__name__
    label_savings_rel = 'savings_rel_' + standard.__name__
    df[label_savings_abs] = df['total_power_in'] - df[label_power] 
    df[label_savings_rel] = df[label_savings_abs] / df['total_power_in'] 

    # .. compute the global savings
    print('For using only one PSU of at least standard ',standard.__name__, '\\\\')

    # .. 1. from the first step
    savings_total_abs = df['savings_abs'].sum(axis=0)
    savings_total_rel = savings_total_abs / (df['median_power_PSU1'].sum(axis=0) + df['median_power_PSU2'].sum(axis=0))
    print("using only one {}\\% ({} W)\\\\".format(
    int(100*savings_total_rel),
    int(savings_total_abs)
    ))
    
    # .. 2. from this step alone
    savings_total_abs = df['savings_abs_'+standard.__name__].sum(axis=0)
    savings_total_rel = savings_total_abs / (df['median_power_PSU1'].sum(axis=0) + df['median_power_PSU2'].sum(axis=0))
    print("using a better one {}\\% ({} W)\\\\".format(
    int(100*savings_total_rel),
    int(savings_total_abs)
    ))

    # .. 3. all together
    savings_total_abs = df['savings_abs_'+standard.__name__].sum(axis=0) + df['savings_abs'].sum(axis=0)
    savings_total_rel = savings_total_abs / (df['median_power_PSU1'].sum(axis=0) + df['median_power_PSU2'].sum(axis=0))
    print("total: {}\\% ({} W)\n".format(
    int(100*savings_total_rel),
    int(savings_total_abs)
    ))

    tex_output += '& {}\\% ({} W)'.format(
        int(100*savings_total_rel),
        int(savings_total_abs)
        )
    
tex_output += '\\\\'
print(tex_output)

df


Analysis type: max_1
For using only one PSU of at least standard  b \\
using only one 5\% (1130 W)\\
using a better one 0\% (138 W)\\
total: 5\% (1269 W)

For using only one PSU of at least standard  s \\
using only one 5\% (1130 W)\\
using a better one 1\% (284 W)\\
total: 6\% (1415 W)

For using only one PSU of at least standard  g \\
using only one 5\% (1130 W)\\
using a better one 1\% (407 W)\\
total: 7\% (1538 W)

For using only one PSU of at least standard  p \\
using only one 5\% (1130 W)\\
using a better one 2\% (526 W)\\
total: 7\% (1657 W)

For using only one PSU of at least standard  t \\
using only one 5\% (1130 W)\\
using a better one 3\% (802 W)\\
total: 9\% (1933 W)

& 5\% (1269 W)& 6\% (1415 W)& 7\% (1538 W)& 7\% (1657 W)& 9\% (1933 W)\\


,router_id,router_model,mean_power_PSU1,mean_power_PSU2,median_power_PSU1,median_power_PSU2,PSU_capacity,efficiency_PSU1,efficiency_PSU2,inlet_temperature,load_PSU1,load_PSU2,total_power_out,total_load,efficiency_PSU1_total_load,efficiency_PSU2_total_load,max_efficiency,total_power_in,savings_abs,savings_rel,efficiency_b,median_power_b,savings_abs_b,savings_rel_b,efficiency_s,median_power_s,savings_abs_s,savings_rel_s,efficiency_g,median_power_g,savings_abs_g,savings_rel_g,efficiency_p,median_power_p,savings_abs_p,savings_rel_p,efficiency_t,median_power_t,savings_abs_t,savings_rel_t
0,swiFN1,NCS-55A1-24H,191.171476,180.411720,191.020644,180.411000,1100,0.808791,0.839479,NaN,0.156268,0.146749,305.947134,0.278134,0.842789,0.878598,0.878598,348.222031,23.209613,0.062487,0.878598,348.222031,0.000000,0.000000,0.881300,347.154493,1.067537,0.003066,0.911300,335.726162,12.495869,0.035885,0.933872,327.611285,20.610746,0.059189,0.972056,314.742420,33.479611,0.096144
1,swiWY1,NCS-55A1-24H,194.310960,165.750012,194.311000,165.499000,1100,0.878525,0.922219,NaN,0.159239,0.133289,323.333346,0.293939,0.912244,0.971484,0.971484,332.824224,26.985776,0.075000,0.971484,332.824224,0.000000,0.000000,0.971484,332.824224,0.000000,0.000000,0.971484,332.824224,0.000000,0.000000,0.971484,332.824224,0.000000,0.000000,0.973360,332.182659,0.641564,0.001928
2,swiES3,NCS-55A1-24H,191.890173,166.704280,191.782000,164.063000,1100,0.938731,0.857143,NaN,0.156954,0.132000,320.657109,0.291506,0.973480,0.907105,0.973480,329.392705,26.452295,0.074337,0.973480,329.392705,0.000000,0.000000,0.973480,329.392705,0.000000,0.000000,0.973480,329.392705,0.000000,0.000000,0.973480,329.392705,0.000000,0.000000,0.973480,329.392705,0.000000,0.000000
3,swiUS1,NCS-55A1-24H,180.736509,167.841851,180.950930,167.873916,1100,0.872845,0.790150,NaN,0.147239,0.135425,290.587640,0.264171,0.910422,0.835413,0.910422,319.179231,29.645615,0.084987,0.910422,319.179231,0.000000,0.000000,0.910422,319.179231,0.000000,0.000000,0.910422,319.179231,0.000000,0.000000,0.932643,311.574391,7.604839,0.023826,0.970826,299.319970,19.859260,0.062220
4,swiTR3,NCS-55A1-24H,212.974344,180.248504,213.636000,180.261268,1100,0.883041,0.869099,NaN,0.176869,0.146612,345.314169,0.313922,0.909058,0.911140,0.911140,378.991512,14.905755,0.037842,0.911140,378.991512,0.000000,0.000000,0.911140,378.991512,0.000000,0.000000,0.914133,377.750310,1.241202,0.003275,0.936706,368.647285,10.344228,0.027294,0.974889,354.208581,24.782932,0.065392
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102,swiWB1,NCS-55A1-48Q6H,186.161235,170.581386,185.614968,171.750000,1100,0.925602,0.812227,NaN,0.151409,0.138907,311.305539,0.283005,0.962616,0.856874,0.962616,323.395348,33.969619,0.095056,0.962616,323.395348,0.000000,0.000000,0.962616,323.395348,0.000000,0.000000,0.962616,323.395348,0.000000,0.000000,0.962616,323.395348,0.000000,0.000000,0.972485,320.113612,3.281736,0.010148
103,swiUN540,N540-24Z8Q2C-M,79.681341,79.645633,79.698000,79.665019,400,0.679551,0.693708,NaN,0.181931,0.181848,109.423071,0.273558,0.699917,0.714114,0.714114,153.229178,6.133841,0.038490,0.840897,130.126648,23.102530,0.150771,0.880897,124.217822,29.011357,0.189333,0.910897,120.126760,33.102418,0.216032,0.933469,117.221909,36.007270,0.234990,0.971653,112.615418,40.613761,0.265052
104,swiOR1,WS-C6504-E,164.589201,172.267520,164.781000,169.380210,2700,NaN,NaN,NaN,0.050601,0.052077,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
105,swiOT2,8201-24H8FH,110.206985,156.885401,112.000000,158.175901,2000,0.545455,0.645652,26.0,0.046264,0.066433,163.217524,0.081609,0.569730,0.656075,0.656075,248.778823,21.397078,0.079197,0.757845,215.370730,33.408093,0.134288,0.797845,204.573101,44.205723,0.177691,0.827845,197.159639,51.619184,0.207490,0.850417,191.926404,56.852419,0.228526,0.888600,183.679315,65.099